# Running a language model on your own Mac

This notebook loads **Phi-3-mini** — a real 3.8-billion-parameter language model — and runs it
**entirely on your own machine**. No API key, no account, no cloud. Once the model file is
downloaded you could unplug the internet and it would still work.

Three separate pieces are involved, and it helps to keep them apart:

| Piece | What it is | Think of it as |
|---|---|---|
| **The `.gguf` file** | Phi-3-mini's weights in one 7.6 GB file | the model's *brain*, sitting on disk |
| **llama.cpp** | A C++ program that runs GGUF models fast | the *engine* that reads the brain |
| **LangChain** | Python library for prompts and chains | the *steering wheel* you actually touch |

Read the stack below from the bottom up: your GPU does the real work, and each layer above it is a
friendlier wrapper around the one below.

<svg width="100%" viewBox="0 0 720 440" xmlns="http://www.w3.org/2000/svg" font-family="-apple-system, BlinkMacSystemFont, Segoe UI, sans-serif">
<defs><marker id="ar1" markerWidth="9" markerHeight="9" refX="7" refY="4.5" orient="auto"><path d="M0,1 L7,4.5 L0,8 Z" fill="#94A3B8"/></marker></defs>
  <rect x="30" y="18" width="430" height="50" rx="8" fill="#DCE6FF" stroke="#4C7EF3" stroke-width="1.5"/>
  <text x="46" y="40" font-size="14" font-weight="600" fill="#1E293B">Your Python code</text>
  <text x="46" y="58" font-size="11.5" fill="#475569">basic_chain.invoke({"input_prompt": "..."})</text>
  <rect x="30" y="88" width="430" height="50" rx="8" fill="#EDE4FF" stroke="#8B5CF6" stroke-width="1.5"/>
  <text x="46" y="110" font-size="14" font-weight="600" fill="#1E293B">LangChain</text>
  <text x="46" y="128" font-size="11.5" fill="#475569">PromptTemplate  |  LlamaCpp  — builds the prompt, calls the model</text>
  <rect x="30" y="158" width="430" height="50" rx="8" fill="#D6F5EA" stroke="#10B981" stroke-width="1.5"/>
  <text x="46" y="180" font-size="14" font-weight="600" fill="#1E293B">llama-cpp-python</text>
  <text x="46" y="198" font-size="11.5" fill="#475569">the pip package — Python bindings</text>
  <rect x="30" y="228" width="430" height="50" rx="8" fill="#D6F5EA" stroke="#10B981" stroke-width="1.5"/>
  <text x="46" y="250" font-size="14" font-weight="600" fill="#1E293B">llama.cpp</text>
  <text x="46" y="268" font-size="11.5" fill="#475569">C++ engine — does the actual matrix maths</text>
  <rect x="30" y="298" width="430" height="50" rx="8" fill="#FFF0D6" stroke="#F59E0B" stroke-width="1.5"/>
  <text x="46" y="320" font-size="14" font-weight="600" fill="#1E293B">Metal</text>
  <text x="46" y="338" font-size="11.5" fill="#475569">Apple's GPU API — what n_gpu_layers=-1 switches on</text>
  <rect x="30" y="368" width="430" height="50" rx="8" fill="#E7ECF2" stroke="#64748B" stroke-width="1.5"/>
  <text x="46" y="390" font-size="14" font-weight="600" fill="#1E293B">Your Mac's GPU</text>
  <text x="46" y="408" font-size="11.5" fill="#475569">Apple Silicon — unified memory shared with the CPU</text>
  <path d="M245,68 L245,86" stroke="#94A3B8" stroke-width="1.5" marker-end="url(#ar1)"/>
  <path d="M245,138 L245,156" stroke="#94A3B8" stroke-width="1.5" marker-end="url(#ar1)"/>
  <path d="M245,208 L245,226" stroke="#94A3B8" stroke-width="1.5" marker-end="url(#ar1)"/>
  <path d="M245,278 L245,296" stroke="#94A3B8" stroke-width="1.5" marker-end="url(#ar1)"/>
  <path d="M245,348 L245,366" stroke="#94A3B8" stroke-width="1.5" marker-end="url(#ar1)"/>
  <rect x="520" y="205" width="170" height="96" rx="8" fill="#FFF0D6" stroke="#F59E0B" stroke-width="1.5"/>
  <text x="605" y="232" font-size="13" font-weight="600" fill="#1E293B" text-anchor="middle">Phi-3-mini-4k</text>
  <text x="605" y="252" font-size="11.5" fill="#475569" text-anchor="middle">fp16 GGUF file</text>
  <text x="605" y="270" font-size="11.5" fill="#475569" text-anchor="middle">7.6 GB on disk</text>
  <text x="605" y="288" font-size="11.5" fill="#475569" text-anchor="middle">195 tensors</text>
  <path d="M518,253 L462,253" stroke="#94A3B8" stroke-width="1.5" marker-end="url(#ar1)"/>
  <text x="605" y="190" font-size="11" fill="#94A3B8" text-anchor="middle">the weights, loaded once</text>
</svg>

**What you will build by the end:** a single question-and-answer call, then a *chain* that pipes a
prompt into the model, then three chains wired together that invent a story title, a character, and
a full story — each step feeding the next.

## Loading a LLM model


### Step 1 — get the model file

A **GGUF file** is a single self-contained file holding everything the model needs: the vocabulary,
the architecture description, and the weights. One file, no folder of shards.

<svg width="100%" viewBox="0 0 720 205" xmlns="http://www.w3.org/2000/svg" font-family="-apple-system, BlinkMacSystemFont, Segoe UI, sans-serif">
  <text x="30" y="22" font-size="12.5" font-weight="600" fill="#94A3B8">What is inside one .gguf file</text>
  <rect x="30" y="38" width="70" height="66" rx="5" fill="#DCE6FF" stroke="#4C7EF3" stroke-width="1.5"/>
  <text x="65" y="68" font-size="12" font-weight="600" fill="#1E293B" text-anchor="middle">GGUF</text>
  <text x="65" y="86" font-size="10.5" fill="#475569" text-anchor="middle">magic</text>
  <rect x="104" y="38" width="60" height="66" rx="5" fill="#DCE6FF" stroke="#4C7EF3" stroke-width="1.5"/>
  <text x="134" y="68" font-size="12" font-weight="600" fill="#1E293B" text-anchor="middle">v3</text>
  <text x="134" y="86" font-size="10.5" fill="#475569" text-anchor="middle">version</text>
  <rect x="168" y="38" width="150" height="66" rx="5" fill="#EDE4FF" stroke="#8B5CF6" stroke-width="1.5"/>
  <text x="243" y="64" font-size="12" font-weight="600" fill="#1E293B" text-anchor="middle">metadata</text>
  <text x="243" y="81" font-size="10.5" fill="#475569" text-anchor="middle">vocab, architecture,</text>
  <text x="243" y="95" font-size="10.5" fill="#475569" text-anchor="middle">195 tensor names</text>
  <rect x="322" y="38" width="368" height="66" rx="5" fill="#D6F5EA" stroke="#10B981" stroke-width="1.5"/>
  <text x="506" y="66" font-size="12.5" font-weight="600" fill="#1E293B" text-anchor="middle">tensor data — the actual weights</text>
  <text x="506" y="86" font-size="11" fill="#475569" text-anchor="middle">7.55 GB, about 99% of the file</text>
  <path d="M30,118 L318,118" stroke="#94A3B8" stroke-width="1"/>
  <text x="174" y="138" font-size="11" fill="#94A3B8" text-anchor="middle">header — only ~90 MB</text>
  <path d="M322,118 L690,118" stroke="#94A3B8" stroke-width="1"/>
  <text x="506" y="138" font-size="11" fill="#94A3B8" text-anchor="middle">everything the model actually knows</text>
  <rect x="30" y="154" width="660" height="38" rx="6" fill="#FFF0D6" stroke="#F59E0B" stroke-width="1.5"/>
  <text x="46" y="178" font-size="11.5" fill="#475569">A part-downloaded file keeps a valid header, so it still looks like a model — until it fails to load. Always check the size.</text>
</svg>

Notice how lopsided it is — the header is tiny, the weights are ~99%. That is why checking the
final size matters: an interrupted download leaves a *valid-looking* header, so the file still
opens and still reports its tensor count while the actual knowledge is missing.

**Why the `!wget` line is commented out.** That was the original line from the book, and it fails
here because **macOS does not ship `wget`** — those notebooks were written for Google Colab, which
runs Linux. macOS gives you `curl` instead, which is why the working line uses it. The `-L` matters:
Hugging Face redirects to a CDN, and without `-L` you would save a 1 KB redirect stub.

⚠️ **The output below this cell shows the download failing** with `KeyboardInterrupt` and then
`OSError: [Errno 5] Input/output error`, after reaching about 74 MB of 7289 MB. Errno 5 here is the
notebook losing its connection to the shell process, not a problem with the URL. If it happens again:

- Run the download in a **Terminal window** rather than in a notebook cell — a 7.6 GB download is a
  long time to hold a notebook cell open, and any kernel hiccup kills it.
- Or use `hf_hub_download`, which **resumes** from where it stopped instead of starting over:

```python
from huggingface_hub import hf_hub_download
hf_hub_download("microsoft/Phi-3-mini-4k-instruct-gguf",
                "Phi-3-mini-4k-instruct-fp16.gguf", local_dir=".")
```

The file is ~7.6 GB, so expect several minutes even on a fast connection.

In [6]:
# !wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf
!curl -L -O https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1052  100  1052    0     0   4811      0 --:--:-- --:--:-- --:--:--  4803
  1 7289M    1 74.4M    0     0  14.8M      0  0:08:10  0:00:05  0:08:05 20.1M

OSError: [Errno 5] Input/output error

### Step 2 — install the engine

`llama-cpp-python` is the Python wrapper around llama.cpp, the C++ engine that actually runs the model.

Two things worth knowing on a Mac:

1. **There is no ready-made version for macOS.** pip downloads the source code and *compiles* it on
   your machine, which takes roughly 5–15 minutes and looks frozen while it works. Do not interrupt it.
   Compiling needs Xcode Command Line Tools — install with `xcode-select --install` if it complains.
2. **Metal support is built in automatically** on Apple Silicon, so no special flags are needed here.

⚠️ **If this cell fails with `error: externally-managed-environment`**, that is macOS protecting the
Python that Homebrew installed. The fix is to work inside a **virtual environment**:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install llama-cpp-python
python -m ipykernel install --user --name genai --display-name "GenAI"
```

Then pick that kernel in the kernel picker. Do **not** use `--break-system-packages` — it is the flag
the error message suggests, and the one that can genuinely damage your Python installation.

In [ ]:
!pip install llama-cpp-python

---

### Step 3 — load the model onto the GPU

This is the most important cell in the notebook, and it turns on one argument.

A language model is built from **layers** stacked on top of each other — Phi-3-mini has 33. Each one
can run on either the CPU or the GPU, and `n_gpu_layers` decides which.

<svg width="100%" viewBox="0 0 720 285" xmlns="http://www.w3.org/2000/svg" font-family="-apple-system, BlinkMacSystemFont, Segoe UI, sans-serif">
  <text x="30" y="22" font-size="12.5" font-weight="600" fill="#94A3B8">Where the model's 33 layers actually run</text>
  <rect x="30" y="36" width="320" height="228" rx="8" fill="none" stroke="#94A3B8" stroke-width="1.5" stroke-dasharray="4 3"/>
  <text x="190" y="60" font-size="13" font-weight="600" fill="#1E293B" text-anchor="middle">n_gpu_layers=0</text>
  <text x="190" y="78" font-size="11" fill="#94A3B8" text-anchor="middle">the default — easy to miss</text>
  <rect x="54" y="92" width="272" height="116" rx="6" fill="#E7ECF2" stroke="#64748B" stroke-width="1.5"/>
  <text x="190" y="112" font-size="11.5" font-weight="600" fill="#1E293B" text-anchor="middle">CPU</text>
  <rect x="68" y="124" width="22" height="66" rx="3" fill="#CBD5E1" stroke="#64748B" stroke-width="1"/><rect x="98" y="124" width="22" height="66" rx="3" fill="#CBD5E1" stroke="#64748B" stroke-width="1"/><rect x="128" y="124" width="22" height="66" rx="3" fill="#CBD5E1" stroke="#64748B" stroke-width="1"/><rect x="158" y="124" width="22" height="66" rx="3" fill="#CBD5E1" stroke="#64748B" stroke-width="1"/><rect x="188" y="124" width="22" height="66" rx="3" fill="#CBD5E1" stroke="#64748B" stroke-width="1"/><rect x="218" y="124" width="22" height="66" rx="3" fill="#CBD5E1" stroke="#64748B" stroke-width="1"/><rect x="248" y="124" width="22" height="66" rx="3" fill="#CBD5E1" stroke="#64748B" stroke-width="1"/><rect x="278" y="124" width="22" height="66" rx="3" fill="#CBD5E1" stroke="#64748B" stroke-width="1"/>
  <text x="190" y="202" font-size="10.5" fill="#475569" text-anchor="middle">all 33 layers</text>
  <text x="190" y="236" font-size="12" font-weight="600" fill="#64748B" text-anchor="middle">GPU sits idle</text>
  <text x="190" y="254" font-size="11" fill="#94A3B8" text-anchor="middle">several times slower</text>
  <rect x="370" y="36" width="320" height="228" rx="8" fill="none" stroke="#10B981" stroke-width="1.5"/>
  <text x="530" y="60" font-size="13" font-weight="600" fill="#1E293B" text-anchor="middle">n_gpu_layers=-1</text>
  <text x="530" y="78" font-size="11" fill="#94A3B8" text-anchor="middle">what your code already uses</text>
  <rect x="394" y="92" width="272" height="116" rx="6" fill="#D6F5EA" stroke="#10B981" stroke-width="1.5"/>
  <text x="530" y="112" font-size="11.5" font-weight="600" fill="#1E293B" text-anchor="middle">Metal GPU</text>
  <rect x="408" y="124" width="22" height="66" rx="3" fill="#6EE7B7" stroke="#10B981" stroke-width="1"/><rect x="438" y="124" width="22" height="66" rx="3" fill="#6EE7B7" stroke="#10B981" stroke-width="1"/><rect x="468" y="124" width="22" height="66" rx="3" fill="#6EE7B7" stroke="#10B981" stroke-width="1"/><rect x="498" y="124" width="22" height="66" rx="3" fill="#6EE7B7" stroke="#10B981" stroke-width="1"/><rect x="528" y="124" width="22" height="66" rx="3" fill="#6EE7B7" stroke="#10B981" stroke-width="1"/><rect x="558" y="124" width="22" height="66" rx="3" fill="#6EE7B7" stroke="#10B981" stroke-width="1"/><rect x="588" y="124" width="22" height="66" rx="3" fill="#6EE7B7" stroke="#10B981" stroke-width="1"/><rect x="618" y="124" width="22" height="66" rx="3" fill="#6EE7B7" stroke="#10B981" stroke-width="1"/>
  <text x="530" y="202" font-size="10.5" fill="#475569" text-anchor="middle">all 33 layers</text>
  <text x="530" y="236" font-size="12" font-weight="600" fill="#10B981" text-anchor="middle">offloaded 33/33 layers to GPU</text>
  <text x="530" y="254" font-size="11" fill="#94A3B8" text-anchor="middle">look for this line in the output</text>
</svg>

**The trap:** the default is `0`. Forget this argument and everything still works — no error, no
warning — it is just quietly running on the CPU, several times slower. Almost every "why is Metal
not working?" question comes down to this one missing line. Your code below has `-1`, which is
shorthand for *all of them*, so you are set.

<svg width="100%" viewBox="0 0 720 215" xmlns="http://www.w3.org/2000/svg" font-family="-apple-system, BlinkMacSystemFont, Segoe UI, sans-serif">
  <text x="30" y="22" font-size="12.5" font-weight="600" fill="#94A3B8">Generation speed on an M1 Pro (tokens per second — higher is better)</text>
  <text x="30" y="59" font-size="12" fill="#475569">fp16 on GPU</text><rect x="180" y="44" width="173" height="22" rx="4" fill="#D6F5EA" stroke="#10B981" stroke-width="1.5"/><text x="361" y="60" font-size="12" font-weight="600" fill="#10B981">20.1  measured</text>
  <text x="30" y="101" font-size="12" fill="#475569">fp16 on CPU</text><rect x="180" y="86" width="34" height="22" rx="4" fill="#E7ECF2" stroke="#64748B" stroke-width="1.5"/><text x="222" y="102" font-size="12" font-weight="600" fill="#64748B">~4  estimate</text>
  <text x="30" y="143" font-size="12" fill="#475569">q4 on GPU</text><rect x="180" y="128" width="344" height="22" rx="4" fill="#EDE4FF" stroke="#8B5CF6" stroke-width="1.5"/><text x="532" y="144" font-size="12" font-weight="600" fill="#8B5CF6">~40  estimate</text>
  <line x1="180" y1="158" x2="640" y2="158" stroke="#CBD5E1" stroke-width="1"/>
  <text x="180" y="174" font-size="10.5" fill="#94A3B8">0</text>
  <text x="352" y="174" font-size="10.5" fill="#94A3B8">20</text>
  <text x="524" y="174" font-size="10.5" fill="#94A3B8">40</text>
  <text x="30" y="200" font-size="11" fill="#94A3B8">Only the first bar was measured (250 tokens in 12.42 s on this machine). The other two are estimates.</text>
</svg>

**Reading the other arguments in the cell below:**

| Argument | What it does |
|---|---|
| `model_path=` | Where the `.gguf` file is. Relative to the notebook, so the file must sit in this folder. |
| `n_gpu_layers=-1` | Put every layer on the GPU. |
| `max_tokens=500` | Longest reply it will produce. A token is roughly ¾ of a word. |
| `n_ctx=2048` | **Context window** — prompt and reply combined must fit in this many tokens. |
| `seed=42` | Fixes the randomness so the same prompt gives the same answer. Good for learning. |
| `verbose=False` | Hides llama.cpp's startup log. |

Two notes on this cell specifically:

- **`n_ctx=2048` is half of what this model supports.** It is called Phi-3-mini-**4k** because it was
  trained for 4096 tokens. Raising it to `n_ctx=4096` costs a little more memory and gives you twice
  the room — useful once the story chains later in this notebook start passing long text around.
- **`from langchain import LlamaCpp` is an old import path.** If you get an `ImportError` or a
  deprecation warning, the current location is:
  ```python
  from langchain_community.llms import LlamaCpp
  ```

**What to look for when it loads.** Set `verbose=True` and you should see:

```
ggml_metal_device_init: GPU name:   MTL0 (Apple M1 Pro)
offloaded 33/33 layers to GPU
```

`33/33` means every layer made it. `0/33` means Metal never engaged.

<svg width="100%" viewBox="0 0 720 200" xmlns="http://www.w3.org/2000/svg" font-family="-apple-system, BlinkMacSystemFont, Segoe UI, sans-serif">
  <text x="30" y="22" font-size="12.5" font-weight="600" fill="#94A3B8">16 GB of unified memory while the model is loaded (fp16)</text>
  <rect x="30" y="38" width="660" height="48" rx="6" fill="#F1F5F9" stroke="#94A3B8" stroke-width="1.5"/>
  <rect x="31" y="39" width="313" height="46" fill="#D6F5EA"/>
  <rect x="344" y="39" width="41" height="46" fill="#EDE4FF"/>
  <rect x="385" y="39" width="248" height="46" fill="#E7ECF2"/>
  <text x="187" y="58" font-size="11.5" font-weight="600" fill="#1E293B" text-anchor="middle">model weights</text>
  <text x="187" y="75" font-size="11" fill="#475569" text-anchor="middle">7.6 GB</text>
  <text x="364" y="67" font-size="10.5" fill="#475569" text-anchor="middle">KV</text>
  <text x="509" y="58" font-size="11.5" font-weight="600" fill="#1E293B" text-anchor="middle">macOS, browser, VS Code</text>
  <text x="509" y="75" font-size="11" fill="#475569" text-anchor="middle">~6 GB</text>
  <rect x="30" y="38" width="660" height="48" rx="6" fill="none" stroke="#94A3B8" stroke-width="1.5"/>
  <line x1="554" y1="30" x2="554" y2="94" stroke="#F59E0B" stroke-width="1.5" stroke-dasharray="4 3"/>
  <text x="554" y="24" font-size="10.5" fill="#F59E0B" text-anchor="middle">GPU limit 12.7 GB</text>
  <text x="30" y="112" font-size="11" fill="#94A3B8">0 GB</text>
  <text x="660" y="112" font-size="11" fill="#94A3B8">16 GB</text>
  <rect x="30" y="128" width="660" height="58" rx="6" fill="#FFF0D6" stroke="#F59E0B" stroke-width="1.5"/>
  <text x="46" y="150" font-size="11.5" fill="#475569">This is only used while the kernel is running with the model loaded. Restarting the kernel frees all of it.</text>
  <text x="46" y="170" font-size="11.5" fill="#475569">The file sitting on disk costs 0 RAM. The q4 version would shrink the green block from 7.6 GB to 2.4 GB.</text>
</svg>

In [ ]:
from langchain import LlamaCpp

# Make sure the model path is correct for your system!
llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1, # indicates that all layers must be using GPU's
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

### Step 4 — ask it something

`.invoke()` is the standard "run this" method across LangChain — the same word works on a model, a
prompt, or a whole chain, which is what makes them stackable.

This first call takes a few seconds: the model has to load into memory before it can answer. Later
calls in the same session are much quicker because it is already there.

Note that the question here is passed as **plain text**, with no `<|user|>` markers around it. It
works, but the reply is often rambly. The next section fixes that.

In [ ]:
# Generate a conversation and ask a basic question
llm.invoke( "What is 1 + 1?")

## Chain

### Step 5 — a prompt template

Phi-3 was trained on a specific **chat format**, and it behaves noticeably better when you use it
instead of raw text.

<svg width="100%" viewBox="0 0 720 245" xmlns="http://www.w3.org/2000/svg" font-family="-apple-system, BlinkMacSystemFont, Segoe UI, sans-serif">
<defs><marker id="ar2" markerWidth="9" markerHeight="9" refX="7" refY="4.5" orient="auto"><path d="M0,1 L7,4.5 L0,8 Z" fill="#94A3B8"/></marker></defs>
  <text x="30" y="22" font-size="12.5" font-weight="600" fill="#94A3B8">Phi-3 does not read plain text — it expects its own chat format</text>
  <rect x="30" y="38" width="200" height="52" rx="6" fill="#DCE6FF" stroke="#4C7EF3" stroke-width="1.5"/>
  <text x="130" y="60" font-size="11.5" font-weight="600" fill="#1E293B" text-anchor="middle">what you want to ask</text>
  <text x="130" y="79" font-size="11" fill="#475569" text-anchor="middle">"what is 1+1"</text>
  <path d="M232,64 L268,64" stroke="#94A3B8" stroke-width="1.5" marker-end="url(#ar2)"/>
  <rect x="272" y="28" width="418" height="72" rx="6" fill="#EDE4FF" stroke="#8B5CF6" stroke-width="1.5"/>
  <text x="288" y="50" font-size="11.5" font-family="ui-monospace, monospace" fill="#7C3AED">&lt;s&gt;&lt;|user|&gt;</text>
  <text x="288" y="70" font-size="11.5" font-family="ui-monospace, monospace" fill="#1E293B">what is 1+1</text>
  <text x="288" y="90" font-size="11.5" font-family="ui-monospace, monospace" fill="#7C3AED">&lt;|end|&gt;  &lt;|assistant|&gt;</text>
  <path d="M481,102 L481,128" stroke="#94A3B8" stroke-width="1.5" marker-end="url(#ar2)"/>
  <rect x="272" y="132" width="418" height="44" rx="6" fill="#D6F5EA" stroke="#10B981" stroke-width="1.5"/>
  <text x="481" y="159" font-size="12" font-weight="600" fill="#1E293B" text-anchor="middle">the model writes its reply, one token at a time</text>
  <path d="M481,178 L481,198" stroke="#94A3B8" stroke-width="1.5" marker-end="url(#ar2)"/>
  <rect x="272" y="202" width="418" height="38" rx="6" fill="#FFF0D6" stroke="#F59E0B" stroke-width="1.5"/>
  <text x="481" y="226" font-size="11.5" fill="#475569" text-anchor="middle">&lt;|end|&gt; marks where the answer stops</text>
  <text x="130" y="140" font-size="11" fill="#94A3B8" text-anchor="middle">Those &lt;|...|&gt; markers are</text>
  <text x="130" y="157" font-size="11" fill="#94A3B8" text-anchor="middle">special tokens Phi-3 was</text>
  <text x="130" y="174" font-size="11" fill="#94A3B8" text-anchor="middle">trained on. &lt;s&gt; just means</text>
  <text x="130" y="191" font-size="11" fill="#94A3B8" text-anchor="middle">"start of text".</text>
</svg>

A **prompt template** is a reusable sentence with `{blanks}` in it. You write the wording once —
including those fiddly `<|user|>` markers — then fill the blanks with different values each time.
Like an f-string, but as an object you can pass around, reuse, and chain onto a model.

- `template = """..."""` — the triple quotes let the string span several lines.
- `{input_prompt}` — the blank to fill.
- `input_variables=["input_prompt"]` — tells LangChain which names to expect.

Because the markers live inside the template, you never have to remember them again.

In [ ]:
# from langchain import PromptTemplate
from langchain_core.prompts import PromptTemplate

# create a prompt template with the "input_prompt" variable
template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables =["input_prompt"]
)

### Step 6 — connect them into a chain

<svg width="100%" viewBox="0 0 720 175" xmlns="http://www.w3.org/2000/svg" font-family="-apple-system, BlinkMacSystemFont, Segoe UI, sans-serif">
<defs><marker id="ar3" markerWidth="9" markerHeight="9" refX="7" refY="4.5" orient="auto"><path d="M0,1 L7,4.5 L0,8 Z" fill="#94A3B8"/></marker></defs>
  <text x="30" y="22" font-size="12.5" font-weight="600" fill="#94A3B8">basic_chain = prompt | llm</text>
  <rect x="30" y="38" width="150" height="76" rx="6" fill="#DCE6FF" stroke="#4C7EF3" stroke-width="1.5"/>
  <text x="105" y="62" font-size="11.5" font-weight="600" fill="#1E293B" text-anchor="middle">your value</text>
  <text x="105" y="84" font-size="10.5" font-family="ui-monospace, monospace" fill="#475569" text-anchor="middle">input_prompt =</text>
  <text x="105" y="100" font-size="10.5" font-family="ui-monospace, monospace" fill="#475569" text-anchor="middle">"what is 1+1"</text>
  <path d="M182,76 L218,76" stroke="#94A3B8" stroke-width="1.5" marker-end="url(#ar3)"/>
  <rect x="222" y="38" width="180" height="76" rx="6" fill="#EDE4FF" stroke="#8B5CF6" stroke-width="1.5"/>
  <text x="312" y="62" font-size="11.5" font-weight="600" fill="#1E293B" text-anchor="middle">prompt</text>
  <text x="312" y="82" font-size="10.5" fill="#475569" text-anchor="middle">drops your value into</text>
  <text x="312" y="99" font-size="10.5" fill="#475569" text-anchor="middle">the chat template</text>
  <path d="M404,76 L440,76" stroke="#94A3B8" stroke-width="1.5" marker-end="url(#ar3)"/>
  <rect x="444" y="38" width="150" height="76" rx="6" fill="#D6F5EA" stroke="#10B981" stroke-width="1.5"/>
  <text x="519" y="62" font-size="11.5" font-weight="600" fill="#1E293B" text-anchor="middle">llm</text>
  <text x="519" y="82" font-size="10.5" fill="#475569" text-anchor="middle">runs it on the</text>
  <text x="519" y="99" font-size="10.5" fill="#475569" text-anchor="middle">Metal GPU</text>
  <path d="M596,76 L632,76" stroke="#94A3B8" stroke-width="1.5" marker-end="url(#ar3)"/>
  <rect x="636" y="54" width="54" height="44" rx="6" fill="#FFF0D6" stroke="#F59E0B" stroke-width="1.5"/>
  <text x="663" y="81" font-size="11" font-weight="600" fill="#1E293B" text-anchor="middle">text</text>
  <text x="360" y="146" font-size="11.5" font-weight="600" fill="#94A3B8" text-anchor="middle">The | pipe means "send the output of the left into the right"</text>
  <text x="360" y="165" font-size="11" fill="#94A3B8" text-anchor="middle">This style is called LCEL — LangChain Expression Language</text>
</svg>

That single `|` is the whole idea. `prompt | llm` means *format the prompt, then hand the result to
the model*. Chains can keep growing — `prompt | llm | parser` would clean up the output too.

In [ ]:
basic_chain = prompt | llm

### Step 7 — run the chain

`.invoke()` takes a **dictionary**, because a template can have several blanks and they need naming.
The keys must match the `input_variables` you declared.

Try editing the text and re-running. Then try adding a second blank to the template above — say
`{style}` — and pass it here as well.

In [ ]:
# Use the chain
basic_chain.invoke(
    {
        "input_prompt":"Hii | my name is Rajia. what is 1+1",
    }
)

# Chain with Multiple Prompts

This is where it gets interesting. Instead of one prompt, you build **three**, and the output of each
becomes an input to the next: a summary produces a title, the title produces a character, and all
three together produce the story.

<svg width="100%" viewBox="0 0 720 300" xmlns="http://www.w3.org/2000/svg" font-family="-apple-system, BlinkMacSystemFont, Segoe UI, sans-serif">
<defs><marker id="ar4" markerWidth="9" markerHeight="9" refX="7" refY="4.5" orient="auto"><path d="M0,1 L7,4.5 L0,8 Z" fill="#94A3B8"/></marker></defs>
  <text x="30" y="22" font-size="12.5" font-weight="600" fill="#94A3B8">llm_chain = title | character | story</text>
  <rect x="24" y="40" width="118" height="66" rx="6" fill="#DCE6FF" stroke="#4C7EF3" stroke-width="1.5"/>
  <text x="83" y="66" font-size="11.5" font-weight="600" fill="#1E293B" text-anchor="middle">your input</text>
  <text x="83" y="86" font-size="10" fill="#475569" text-anchor="middle">"a girl that lost</text>
  <text x="83" y="99" font-size="10" fill="#475569" text-anchor="middle">her mother"</text>
  <path d="M144,73 L168,73" stroke="#94A3B8" stroke-width="1.5" marker-end="url(#ar4)"/>
  <rect x="172" y="40" width="150" height="66" rx="6" fill="#EDE4FF" stroke="#8B5CF6" stroke-width="1.5"/>
  <text x="247" y="64" font-size="12" font-weight="600" fill="#1E293B" text-anchor="middle">title</text>
  <text x="247" y="84" font-size="10.5" fill="#475569" text-anchor="middle">invents a title from</text>
  <text x="247" y="98" font-size="10.5" fill="#475569" text-anchor="middle">the summary</text>
  <path d="M324,73 L348,73" stroke="#94A3B8" stroke-width="1.5" marker-end="url(#ar4)"/>
  <rect x="352" y="40" width="150" height="66" rx="6" fill="#EDE4FF" stroke="#8B5CF6" stroke-width="1.5"/>
  <text x="427" y="64" font-size="12" font-weight="600" fill="#1E293B" text-anchor="middle">character</text>
  <text x="427" y="84" font-size="10.5" fill="#475569" text-anchor="middle">needs summary</text>
  <text x="427" y="98" font-size="10.5" fill="#475569" text-anchor="middle">+ title</text>
  <path d="M504,73 L528,73" stroke="#94A3B8" stroke-width="1.5" marker-end="url(#ar4)"/>
  <rect x="532" y="40" width="160" height="66" rx="6" fill="#D6F5EA" stroke="#10B981" stroke-width="1.5"/>
  <text x="612" y="64" font-size="12" font-weight="600" fill="#1E293B" text-anchor="middle">story</text>
  <text x="612" y="84" font-size="10.5" fill="#475569" text-anchor="middle">needs all three</text>
  <text x="612" y="98" font-size="10.5" fill="#475569" text-anchor="middle">and writes it</text>

  <text x="30" y="146" font-size="12" font-weight="600" fill="#94A3B8">What is being carried along at each step</text>
  <rect x="24" y="158" width="118" height="30" rx="4" fill="#F1F5F9" stroke="#CBD5E1" stroke-width="1"/>
  <text x="83" y="178" font-size="10.5" font-family="ui-monospace, monospace" fill="#475569" text-anchor="middle">summary</text>
  <rect x="172" y="158" width="150" height="30" rx="4" fill="#F1F5F9" stroke="#CBD5E1" stroke-width="1"/>
  <text x="247" y="178" font-size="10.5" font-family="ui-monospace, monospace" fill="#475569" text-anchor="middle">summary + title</text>
  <rect x="352" y="158" width="150" height="30" rx="4" fill="#F1F5F9" stroke="#CBD5E1" stroke-width="1"/>
  <text x="427" y="178" font-size="10" font-family="ui-monospace, monospace" fill="#475569" text-anchor="middle">+ character</text>
  <rect x="532" y="158" width="160" height="30" rx="4" fill="#D6F5EA" stroke="#10B981" stroke-width="1"/>
  <text x="612" y="178" font-size="10" font-family="ui-monospace, monospace" fill="#475569" text-anchor="middle">+ story</text>

  <rect x="24" y="208" width="668" height="80" rx="6" fill="#F1F5F9" stroke="#94A3B8" stroke-width="1.5"/>
  <text x="42" y="232" font-size="11.5" fill="#475569">Each LLMChain returns a dictionary holding everything it was given plus its own output_key.</text>
  <text x="42" y="252" font-size="11.5" fill="#475569">So the dictionary grows as it flows right, and each step can reach back for anything an earlier step produced.</text>
  <text x="42" y="274" font-size="11.5" fill="#475569">That is why output_key="title" on one chain matches {title} in the next chain's template.</text>
</svg>

**How the data actually flows.** Each `LLMChain` returns a dictionary containing everything it was
handed *plus* its own result under `output_key`. So the dictionary grows as it moves right, and each
step can reach back for anything an earlier one produced. That is why `output_key="title"` on one
chain lines up with `{title}` in the next chain's template — the names have to match exactly.

⚠️ **Two compatibility notes for these cells:**

- **`from langchain import LLMChain` is an old import path.** If it errors, use
  `from langchain.chains import LLMChain`. `LLMChain` is also deprecated in recent LangChain in
  favour of plain LCEL pipes (`prompt | llm`), so a newer tutorial will look different.
- **Watch the template in the third cell.** The story prompt ends with `Only return the` — the
  sentence is cut off mid-word. The model will still produce something, but it is being given an
  instruction that stops halfway. `Only return the story.` is presumably what was meant.

In [ ]:
from langchain import LLMChain

# create a chain for the titke of our story
template=""" <s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>
"""

title_prompt = PromptTemplate(template=template, input_variables=["summary"])
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

Running one link on its own is a good habit — check each stage produces something sensible before
wiring them together. The result comes back as a dictionary, so you will see both the summary you
passed in and the `title` that came out.

In [ ]:
title.invoke({"summary": "a girl that lost her mother"})

The character chain declares **two** input variables, `summary` and `title`. It can only run once
something has produced a title — which is exactly why the order in the pipe matters.

In [ ]:
# Create a chain for the character description using the summary and title
template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|>
<|assistant|>
 """

character_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title"]
)
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

In [ ]:
# Create a chain for the story using the summary, title, and character description

template = """ <s><|user|>
Create a story about {summary} with the title {title}. The main chararcter is: {character}. Only return the 
<|assistant|>
"""
story_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title", "character"]
)
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

### Wiring all three together

One line builds the whole pipeline. Read `title | character | story` left to right: each stage runs,
adds its result to the dictionary, and passes the whole thing along.

Because three separate generations happen back to back, this cell takes roughly three times as long
as a single call. That is expected — watch the story get built up in stages.

In [ ]:
# Combining all three together
llm_chain = title | character | story

### The final result

The output is the accumulated dictionary: your original summary, the invented `title`, the
`character` description, and the finished `story`. Everything each stage produced is still in there,
which makes it easy to see how the model got where it did.

Try `llm_chain.invoke("a robot who wants to learn painting")` and watch all three stages adapt.

In [ ]:
llm_chain.invoke("a girl that lost her mother")

---

## What to remember

| Thing | Why it matters |
|---|---|
| `n_gpu_layers=-1` | Turns on the Metal GPU. Default is `0` = CPU-only, silently slow. |
| `offloaded 33/33 layers to GPU` | The line that proves the GPU is being used. |
| `n_ctx` | Prompt **and** reply must fit inside it. This model supports 4096. |
| `<s><\|user\|> ... <\|end\|><\|assistant\|>` | Phi-3's chat format. Bake it into the template once. |
| `prompt \| llm` | The pipe. Output of the left becomes input of the right. |
| `output_key` | How one chain's result becomes the next chain's `{variable}`. |
| macOS has no `wget` | Colab notebooks assume Linux. Use `curl -L -O`, or `hf_hub_download` to resume. |
| `externally-managed-environment` | Homebrew Python blocking pip. Work inside a virtual environment. |

**Memory:** the model holds ~7.6 GB of RAM for as long as this kernel is alive. Restarting or
shutting down the kernel releases it immediately. The file on disk costs no RAM at all.

**If things get slow** partway through a session, you may have loaded the model more than once.
Restart the kernel and run the load cell a single time.